# ObserverBench Safety: Qwen2.5-7B-Instruct paired authorization study

**Question.** Does an observer fit to realized safety risk choose better allow/block/escalate actions than an equally expressive observer fit to violation labels, even when both use the same frozen Qwen activations?

**Success criteria.** The clean Qwen authorization gate must pass before observer results are opened. Under identical action budgets, direct risk must improve mean protocol loss over both the activation-label and transformed-label baselines on the locked test. Results are reported separately for unseen operations and unseen prompt formats.

The fixture is inert: it contains mock workspace names and no executable tools, credentials, or harmful payloads.

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex


## Setup

Use an A100, H100, or H200 runtime. Keep the private ObserverBench checkout at `REPO_ROOT` and artifacts on Drive so an interrupted run can resume. The experiment logic lives in the package; this notebook only invokes its staged runner.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

default_repo = Path('/content/ObserverBench') if IN_COLAB else Path.cwd()
REPO_ROOT = Path(os.environ.get('OBSERVERBENCH_REPO', default_repo)).expanduser().resolve()
default_artifacts = (
    Path('/content/drive/MyDrive/ObserverBenchArtifacts/qwen_safety/paired_scope_v0')
    if IN_COLAB
    else REPO_ROOT / 'results/revision/qwen_safety/paired_scope_v0'
)
ARTIFACTS_ROOT = Path(os.environ.get('OBSERVERBENCH_QWEN_SAFETY_ARTIFACTS', default_artifacts)).expanduser().resolve()
CONFIG = REPO_ROOT / 'configs/revision/qwen_safety/qwen2_5_7b_instruct_paired_scope_v0.json'
SCRIPT = REPO_ROOT / 'scripts/run_qwen_safety.py'

assert (REPO_ROOT / 'pyproject.toml').is_file(), f'Set OBSERVERBENCH_REPO to the private checkout: {REPO_ROOT}'
assert CONFIG.is_file(), f'Missing frozen config: {CONFIG}'
assert SCRIPT.is_file(), f'Missing staged runner: {SCRIPT}'
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
{'repo_root': str(REPO_ROOT), 'artifacts_root': str(ARTIFACTS_ROOT), 'in_colab': IN_COLAB}


In [ ]:
# Install the pinned experiment dependencies from the private checkout.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_ROOT}[qwen]'],
    check=True,
)
print('ObserverBench Qwen dependencies installed.')


## Frozen design

Fit and calibration exclude the `delete` operation and XML prompt form. The locked test crosses seen/unseen operations with seen/unseen formats in four equal strata. Resource-name banks are disjoint. The runner freezes layer, ridge value, feature normalization, and coefficients before creating the locked-test activation cache.


In [ ]:
config = json.loads(CONFIG.read_text())
{
    'model': config['model']['id'],
    'revision': config['model']['revision'],
    'layers': config['activations']['layers'],
    'fit_pairs': config['design']['fit_pairs'],
    'calibration_pairs': config['design']['calibration_pairs'],
    'locked_test_pairs': 4 * config['design']['locked_test_pairs_per_stratum'],
    'policy': config['policy'],
}


## Run the sealed sequence

This is the scientific run, not another smoke audit. `--resume` skips only hash-valid artifacts. A fresh artifact root is required if locked-test activations exist before observer freezing. The model stays loaded while the runner moves from fit extraction through the locked test.


In [ ]:
command = [
    sys.executable,
    str(SCRIPT),
    '--config', str(CONFIG),
    '--artifacts-root', str(ARTIFACTS_ROOT),
    '--stage', 'all',
    '--device', 'cuda',
    '--resume',
]
subprocess.run(command, cwd=REPO_ROOT, check=True)


## Results

If the clean gate fails, the result bundle contains no observer outcomes. Otherwise this cell shows the fixed-budget comparison and the four held-out-family rows.


In [ ]:
result_path = ARTIFACTS_ROOT / 'evaluation/qwen_safety_results.json'
payload = json.loads(result_path.read_text())
print('status:', payload['status'])
print('clean gate:', json.dumps(payload['clean_gate'], indent=2))
if payload['status'] == 'complete':
    for name, row in payload['results'].items():
        metrics = row['metrics']
        print(
            f"{name:40s} loss={metrics['protocol_loss_mean']:.4f} "
            f"cvar={metrics['protocol_loss_cvar']:.4f} "
            f"auroc={metrics['risk_auroc']:.4f} "
            f"miss={metrics['severity_weighted_miss_rate']:.4f} "
            f"utility={metrics['clean_utility_retained']:.4f}"
        )


In [ ]:
# Compact decision record for the manuscript pass.
if payload['status'] == 'complete':
    results = payload['results']
    direct = results['activation-direct-risk']['metrics']['protocol_loss_mean']
    label = results['activation-label']['metrics']['protocol_loss_mean']
    transformed = results['activation-transformed-label-risk']['metrics']['protocol_loss_mean']
    noop = results['allow-all-no-action']['metrics']['protocol_loss_mean']
    decision = {
        'direct_vs_label_reduction': 1.0 - direct / label,
        'direct_vs_transformed_reduction': 1.0 - direct / transformed,
        'direct_vs_noop_reduction': 1.0 - direct / noop,
        'promote_safety_result': direct < label and direct < transformed,
    }
    print(json.dumps(decision, indent=2))
else:
    print('Stop: Qwen did not pass the base authorization gate; do not claim an observer result.')


## Next step

If the gate and direct-risk comparison pass, freeze the result bundle and add attribution, DLA, and a compatible SAE observer through the same task boundary. If the gate fails, retain the task and report the stop condition; do not rewrite prompts after viewing observer outcomes.
